# SadTalker Google Colab Queue Worker

This notebook mounts Google Drive, bootstraps a validated environment, loads SadTalker checkpoints,
runs a pre-flight inference sanity test, and then starts the self-contained queue worker script.

**Stability patch applied** — environment mirrors the proven `SadTalker_Test.ipynb` session.

---

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Worker Configuration Block

Configure folders and scanning parameters. Settings here override defaults without altering worker source code.

In [ ]:
# ── Worker Configuration Section ──────────────────────────────────────────────
DRIVE_ROOT         = "/content/drive/MyDrive/WhiteboardAI"  # Google Drive folder root
QUEUE_DIR          = "avatar_queue"
PROCESSING_DIR     = "avatar_processing"
COMPLETED_DIR      = "avatar_completed"
FAILED_DIR         = "avatar_failed"

HEARTBEAT_INTERVAL = 30.0   # Heartbeat worker.json write interval (seconds)
SCAN_INTERVAL      = 5.0    # Queue checking poll interval (seconds)

WORKSPACE          = "/content/workspace"       # Local temp workspace for fast I/O processing
SADTALKER_DIR      = "/content/SadTalker"       # SadTalker installation directory
WORKER_VERSION     = "1.0.0"

# Source location of colab_sadtalker_worker.py in your Google Drive
WORKER_SCRIPT_DIR  = "/content/drive/MyDrive/WhiteboardAI/system/scripts"

# System folder for environment freeze file
SYSTEM_DIR         = "/content/drive/MyDrive/WhiteboardAI/system"

print("[CONFIG] Worker configuration loaded.")
print(f"  DRIVE_ROOT      : {DRIVE_ROOT}")
print(f"  SADTALKER_DIR   : {SADTALKER_DIR}")
print(f"  WORKSPACE       : {WORKSPACE}")
print(f"  WORKER_VERSION  : {WORKER_VERSION}")

## 3. Environment Bootstrap & Verbose Diagnostics

Prints full environment details before installing any packages. Any anomalies here indicate a broken session.

In [ ]:
import sys
import subprocess
import platform

print("=" * 60)
print("  ENVIRONMENT BOOTSTRAP DIAGNOSTICS")
print("=" * 60)

# Python
print(f"\n[PYTHON]")
print(f"  Version : {sys.version}")
print(f"  Platform: {platform.platform()}")
print(f"  Prefix  : {sys.prefix}")

# PyTorch & CUDA
print(f"\n[PYTORCH / CUDA]")
try:
    import torch
    print(f"  torch version : {torch.__version__}")
    print(f"  CUDA available: {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        print(f"  CUDA version  : {torch.version.cuda}")
        print(f"  cuDNN version : {torch.backends.cudnn.version()}")
        print(f"  Device name   : {torch.cuda.get_device_name(0)}")
        total_mem = torch.cuda.get_device_properties(0).total_memory / (1024**3)
        print(f"  VRAM (total)  : {total_mem:.2f} GB")
    else:
        print("  WARNING: No CUDA device found. Worker requires GPU runtime.")
except ImportError:
    print("  torch: NOT INSTALLED")

# FFmpeg
print(f"\n[FFMPEG]")
try:
    ffmpeg_ver = subprocess.check_output(["ffmpeg", "-version"], stderr=subprocess.STDOUT).decode().splitlines()[0]
    print(f"  {ffmpeg_ver}")
except Exception as e:
    print(f"  ffmpeg: NOT FOUND ({e})")

try:
    ffprobe_ver = subprocess.check_output(["ffprobe", "-version"], stderr=subprocess.STDOUT).decode().splitlines()[0]
    print(f"  {ffprobe_ver}")
except Exception as e:
    print(f"  ffprobe: NOT FOUND ({e})")

# OpenCV
print(f"\n[OPENCV]")
try:
    import cv2
    print(f"  cv2 version: {cv2.__version__}")
except ImportError:
    print("  cv2: NOT INSTALLED")

# Disk Space
print(f"\n[DISK SPACE]")
import shutil
total, used, free = shutil.disk_usage("/content")
print(f"  /content  total={total/(1024**3):.1f}GB  used={used/(1024**3):.1f}GB  free={free/(1024**3):.1f}GB")

total_d, used_d, free_d = shutil.disk_usage("/content/drive")
print(f"  /drive    total={total_d/(1024**3):.1f}GB  used={used_d/(1024**3):.1f}GB  free={free_d/(1024**3):.1f}GB")

print("\n" + "=" * 60)
print("  DIAGNOSTICS COMPLETE")
print("=" * 60)

## 4. Clone Repository & Install Pinned Dependencies

Clones SadTalker and installs a **pinned dependency set** mirroring the validated test environment.
The pinned freeze file is also saved to Google Drive for future reproducibility.

In [ ]:
import os
from pathlib import Path

# Clone SadTalker if not already present
if not Path(SADTALKER_DIR).exists():
    print("[INSTALL] Cloning SadTalker repository...")
    !git clone https://github.com/OpenTalker/SadTalker.git {SADTALKER_DIR}
else:
    print("[INSTALL] SadTalker already cloned. Skipping git clone.")

%cd {SADTALKER_DIR}

# ── Pinned dependency installation (mirrors SadTalker_Test.ipynb proven session) ──
print("\n[INSTALL] Installing pinned SadTalker requirements...")
!pip install -q -r requirements.txt

print("\n[INSTALL] Installing HuggingFace + background removal dependencies...")
!pip install -q \
    transformers==4.40.2 \
    accelerate==0.30.1 \
    safetensors==0.4.3 \
    torchvision \
    diffusers==0.27.2 \
    huggingface_hub==0.22.2

print("\n[INSTALL] Installing background segmentation model dependencies...")
!pip install -q \
    timm==0.9.16 \
    einops==0.7.0 \
    kornia==0.7.2 \
    imageio==2.34.1 \
    imageio-ffmpeg==0.5.1

print("\n[INSTALL] All dependencies installed.")

## 5. Freeze Environment to Drive

Saves the full `pip freeze` snapshot to `WhiteboardAI/system/worker_environment.txt` for future reproducibility.

In [ ]:
from pathlib import Path

system_dir = Path(SYSTEM_DIR)
system_dir.mkdir(parents=True, exist_ok=True)

freeze_path = system_dir / "worker_environment.txt"

import subprocess, datetime
freeze_output = subprocess.check_output(["pip", "freeze"]).decode("utf-8")
header = (
    f"# SadTalker Worker Environment Freeze\n"
    f"# Generated: {datetime.datetime.utcnow().isoformat()}Z\n"
    f"# Worker Version: {WORKER_VERSION}\n"
    f"# Python: {sys.version.split()[0]}\n"
    f"# Platform: {platform.platform()}\n\n"
)

freeze_path.write_text(header + freeze_output, encoding="utf-8")
print(f"[ENV_FREEZE] Environment snapshot saved to: {freeze_path}")
print(f"[ENV_FREEZE] Total packages: {len(freeze_output.splitlines())}")

## 6. Checkpoint Download & Validation

Downloads SadTalker model weights if not cached. Validates **every checkpoint file** before proceeding.
The worker will refuse to start if any checkpoint is missing or corrupted.

In [ ]:
from pathlib import Path
import torch

models_cache = Path("/content/models")
models_cache.mkdir(parents=True, exist_ok=True)

# Symlink SadTalker checkpoints dir → /content/models
checkpoints_link = Path(SADTALKER_DIR) / "checkpoints"
if checkpoints_link.exists() and not checkpoints_link.is_symlink():
    import shutil
    shutil.rmtree(str(checkpoints_link))

if not checkpoints_link.exists():
    !ln -s /content/models {SADTALKER_DIR}/checkpoints
    print("[CHECKPOINTS] Symlinked /content/models → checkpoints/")
else:
    print("[CHECKPOINTS] Symlink already exists.")

# Download if not already cached
safetensors_path = models_cache / "SadTalker_V0.0.2_256.safetensors"
if not safetensors_path.exists():
    print("[CHECKPOINTS] Cache miss — downloading model weights...")
    !bash scripts/download_models.sh
else:
    print("[CHECKPOINTS] Cache hit — skipping download.")

# ── Checkpoint Integrity Validation ──────────────────────────────────────────
print("\n[CHECKPOINT_VALIDATION] Verifying all required checkpoint files...")

REQUIRED_CHECKPOINTS = {
    "SadTalker_V0.0.2_256.safetensors": models_cache / "SadTalker_V0.0.2_256.safetensors",
    "mapping_00109-steps.pth"          : models_cache / "mapping_00109-steps.pth",
    "mapping_00229-steps.pth"          : models_cache / "mapping_00229-steps.pth",
}

all_passed = True
for name, ckpt_path in REQUIRED_CHECKPOINTS.items():
    if not ckpt_path.exists():
        print(f"  ✗ MISSING : {name}")
        all_passed = False
        continue
    try:
        if ckpt_path.suffix == ".safetensors":
            from safetensors import safe_open
            with safe_open(str(ckpt_path), framework="pt", device="cpu") as f:
                key_count = len(list(f.keys()))
            print(f"  ✓ OK      : {name}  ({key_count} tensors)")
        else:
            state = torch.load(str(ckpt_path), map_location="cpu", weights_only=True)
            key_count = len(state) if isinstance(state, dict) else "?"
            print(f"  ✓ OK      : {name}  ({key_count} keys)")
    except Exception as e:
        print(f"  ✗ CORRUPT : {name}  ({e})")
        all_passed = False

if not all_passed:
    raise RuntimeError(
        "[CHECKPOINT_VALIDATION] FAILED — one or more checkpoints are missing or corrupted.\n"
        "Delete /content/models and re-run this cell to force a fresh download."
    )

print("\n[CHECKPOINT_VALIDATION] ALL CHECKPOINTS VALID — proceeding.")

## 7. Inference Sanity Test

Runs a **minimal SadTalker inference** using a tiny synthetic image and 1-second WAV tone before entering
the queue loop. This confirms that:
- SadTalker can load its models on the current GPU
- The full inference pipeline executes without errors
- FFmpeg can encode the output

> **Mirrors the test performed in `SadTalker_Test.ipynb`** to guarantee a known-good environment.

In [ ]:
import os
import shutil
import tempfile
import subprocess
import numpy as np
from pathlib import Path
from PIL import Image

print("[SANITY_TEST] Starting pre-flight inference sanity test...")
sanity_dir = Path("/content/sanity_test")
sanity_dir.mkdir(parents=True, exist_ok=True)

# ── Create synthetic test portrait (256x256 solid colour PNG) ─────────────────
portrait_path = sanity_dir / "test_portrait.png"
if not portrait_path.exists():
    img = Image.fromarray(
        np.full((256, 256, 3), [220, 180, 140], dtype=np.uint8)
    )
    img.save(str(portrait_path))
print(f"[SANITY_TEST] Test portrait: {portrait_path}")

# ── Create 1-second silent WAV ────────────────────────────────────────────────
audio_path = sanity_dir / "test_audio.wav"
if not audio_path.exists():
    subprocess.run([
        "ffmpeg", "-y",
        "-f", "lavfi", "-i", "sine=frequency=440:duration=1",
        "-ar", "16000", "-ac", "1",
        str(audio_path)
    ], check=True, capture_output=True)
print(f"[SANITY_TEST] Test audio   : {audio_path}")

# ── Run SadTalker minimal inference ──────────────────────────────────────────
sanity_output = sanity_dir / "output"
sanity_output.mkdir(parents=True, exist_ok=True)

cmd = [
    "python", str(Path(SADTALKER_DIR) / "inference.py"),
    "--driven_audio",  str(audio_path),
    "--source_image",  str(portrait_path),
    "--result_dir",    str(sanity_output),
    "--preprocess",    "full",
    "--size",          "256",
    "--still"
]

print("[SANITY_TEST] Running SadTalker inference...")
import time
t0 = time.time()
result = subprocess.run(cmd, cwd=SADTALKER_DIR, capture_output=True, text=True)
elapsed = time.time() - t0

if result.returncode != 0:
    print("[SANITY_TEST] STDOUT:", result.stdout[-2000:])
    print("[SANITY_TEST] STDERR:", result.stderr[-2000:])
    raise RuntimeError(
        f"[SANITY_TEST] FAILED — SadTalker inference returned non-zero exit code {result.returncode}.\n"
        "Fix the environment before starting the queue worker."
    )

# Verify output video exists
mp4_files = list(sanity_output.rglob("*.mp4"))
if not mp4_files:
    raise RuntimeError("[SANITY_TEST] FAILED — SadTalker returned success but produced no output video.")

print(f"[SANITY_TEST] Output video : {mp4_files[0]}")
print(f"[SANITY_TEST] Inference time: {elapsed:.1f}s")

# ── FFmpeg WebM encode sanity check ───────────────────────────────────────────
sanity_webm = sanity_dir / "sanity.webm"
ffmpeg_cmd = [
    "ffmpeg", "-y",
    "-i", str(mp4_files[0]),
    "-c:v", "libvpx-vp9",
    "-pix_fmt", "yuva420p",
    "-auto-alt-ref", "0",
    "-b:v", "1M",
    str(sanity_webm)
]
enc_result = subprocess.run(ffmpeg_cmd, capture_output=True, text=True)
if enc_result.returncode != 0:
    raise RuntimeError(f"[SANITY_TEST] FFmpeg VP9 encode FAILED:\n{enc_result.stderr[-1500:]}")

print(f"[SANITY_TEST] FFmpeg VP9 encode: OK ({sanity_webm.stat().st_size // 1024} KB)")

# Cleanup sanity dir
shutil.rmtree(str(sanity_dir))
import torch, gc
torch.cuda.empty_cache()
gc.collect()

print("\n[SANITY_TEST] ✓ ALL CHECKS PASSED — Environment is validated and ready.")
print(f"[SANITY_TEST] Proceeding to start the queue worker...")

## 8. Load and Launch Worker

Imports and starts the worker queue scanner loop. It will:
1. Run built-in self-test (GPU, FFmpeg, Drive mount, checkpoint integrity).
2. Scan and requeue abandoned jobs in `avatar_processing/`.
3. Poll `avatar_queue/` and process each job locally before copying results back to Drive.

In [ ]:
import sys
from pathlib import Path

# Register worker script directory in Python path
for _path in [WORKER_SCRIPT_DIR, "/content"]:
    if _path not in sys.path:
        sys.path.insert(0, _path)

try:
    from colab_sadtalker_worker import ColabWorker
    print("[WORKER] Loaded colab_sadtalker_worker from Drive scripts directory.")
except ImportError as e:
    raise ImportError(
        f"Could not import ColabWorker: {e}\n"
        f"Ensure colab_sadtalker_worker.py is uploaded to: {WORKER_SCRIPT_DIR}"
    )

# Initialize worker
worker = ColabWorker(
    sadtalker_dir=Path(SADTALKER_DIR),
    drive_root=Path(DRIVE_ROOT),
    queue_dir=QUEUE_DIR,
    processing_dir=PROCESSING_DIR,
    completed_dir=COMPLETED_DIR,
    failed_dir=FAILED_DIR,
    heartbeat_interval=HEARTBEAT_INTERVAL,
    scan_interval=SCAN_INTERVAL,
    workspace_root=WORKSPACE,
    worker_version=WORKER_VERSION
)

print("[WORKER] Starting queue loop. Press Stop to perform clean shutdown.")
worker.start()